## fine-tuning

Before starting, set the "runtime" to a T4 GPU (click on the little down arrow on the right side, next to "RAM Disk" icon)

In [ ]:
!pip install trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.4/366.4 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 58.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 36.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 45.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
from transformers import (
    pipeline,
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
)
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("EleutherAI/gpt-neo-125m")
model = AutoModelForCausalLM.from_pretrained("EleutherAI/gpt-neo-125m")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.11M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/357 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.01k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/526M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/119 [00:00<?, ?B/s]

In [ ]:
ds = load_dataset("gofilipa/heritage_gender")

README.md:   0%|          | 0.00/36.0 [00:00<?, ?B/s]

heritage_foundation_gender.txt:   0%|          | 0.00/607k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/4462 [00:00<?, ? examples/s]

In [ ]:
ds

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 4462
    })
})

In [ ]:
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

training_params = SFTConfig(
    output_dir="~/results",
    per_device_train_batch_size=1,  # Reduce from 4 to 1
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,  # Increase to maintain effective batch size
    num_train_epochs = 1, # how many times we iterate over the dataset as a whole
    learning_rate = 2e-4, # how many "steps" we take in adjusting the parameters to make up for loss
    weight_decay = 0.001, # way of regularizing the parameters
    dataset_text_field = "text"[:500],
    report_to="none" # this is a new param, to avoid a login to W&B
)


trainer = SFTTrainer(
    model = model,
    train_dataset = ds['train'],
    processing_class = tokenizer,
    args = training_params
)

Converting train dataset to ChatML:   0%|          | 0/4462 [00:00<?, ? examples/s]

Adding EOS to train dataset:   0%|          | 0/4462 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/4462 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/4462 [00:00<?, ? examples/s]

In [ ]:
trainer.train()

Step,Training Loss
500,3.658200
1000,3.433600


TrainOutput(global_step=1116, training_loss=3.5219925911195817, metrics={'train_runtime': 322.5022, 'train_samples_per_second': 13.836, 'train_steps_per_second': 3.46, 'total_flos': 67084818877440.0, 'train_loss': 3.5219925911195817})

In [ ]:
trainer.model.save_pretrained("models")
trainer.tokenizer.save_pretrained("models")

Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.


('models/tokenizer_config.json',
 'models/special_tokens_map.json',
 'models/vocab.json',
 'models/merges.txt',
 'models/added_tokens.json',
 'models/tokenizer.json')

In [ ]:
model = AutoModelForCausalLM.from_pretrained("models")
tokenizer = AutoTokenizer.from_pretrained("models")

In [ ]:
pipe = pipeline('text-generation', model=model, tokenizer=tokenizer, max_length=50)

Device set to use cuda:0


In [ ]:
pipe("Gender identity is defined as")

[{'generated_text': 'Gender identity is defined as the\xa0gender assigned to a person by biological sex and the biological sex assigned to a person by sex or, for biological sex, gender identity.'}]

In [ ]:
pipe("The right of")

[{'generated_text': 'The right of parents to protect their children is only now being recognized.'}]

In [ ]:
pipe("Transgender")

[{'generated_text': 'Transgender policies have not been completely successful.'}]

In [ ]:
pipe("Transgender students")

[{'generated_text': 'Transgender students, and most others in the classroom, have the right to participate in the classroom.'}]